In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from nqbreakout.data.loader import load_nq_1min, resample_to_5min, load_qqq_daily
from nqbreakout.strategy.config import load_strategy_config, load_instrument_spec
from nqbreakout.strategy.core import run_backtest
from nqbreakout.strategy.costs import LEGACY_COST_MODEL
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print('repo:', REPO)

In [ ]:
# Carica i dati NQ 1-min da parquet (skip se non ancora materializzato).
nq_path = REPO / 'data' / 'nq_1min.parquet'
if not nq_path.exists():
    raise FileNotFoundError(
        f'NQ parquet non trovato a {nq_path}. Esegui scripts/convert_to_parquet.py.'
    )
df_1min = load_nq_1min(nq_path)
print('rows:', len(df_1min), 'date range:', df_1min['date'].min(), '->', df_1min['date'].max())

In [ ]:
df_5min = resample_to_5min(df_1min)
print('5-min bars:', len(df_5min))
df_5min.head(3)

In [ ]:
cfg = load_strategy_config(REPO / 'configs' / 'strategy_default.yaml')
cfg

In [ ]:
inst = load_instrument_spec('NQ')
inst

In [ ]:
result = run_backtest(df_5min, cfg, inst, cost_model=LEGACY_COST_MODEL)
print('n trades:', len(result.trades))
print('meta:', result.meta)

In [ ]:
trades = result.trades
if not trades.empty:
    win_rate = (trades['pnl'] > 0).mean()
    print(f'count          : {len(trades)}')
    print(f'total pnl      : ${trades["pnl"].sum():,.2f}')
    print(f'mean pnl       : ${trades["pnl"].mean():,.2f}')
    print(f'mean pnl/contr : ${trades["pnl_per_contract_net"].mean():,.2f}')
    print(f'win rate       : {win_rate:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trades['exit_time'], trades['account_value'] / 1e6, color='blue')
ax.set_xlabel('Exit time')
ax.set_ylabel('Account value ($M)')
ax.set_title('Equity curve')
ax.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(trades['pnl'], bins=60, color='teal', edgecolor='black')
ax.set_xlabel('Trade PnL ($)')
ax.set_ylabel('Count')
ax.set_title('Trade PnL distribution')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
out_path = REPO / 'reports' / 'notebook_run.parquet'
out_path.parent.mkdir(parents=True, exist_ok=True)
trades.to_parquet(out_path, index=False)
print('saved:', out_path)